## Pytorch

As we build towards more computationally expensive models (like deep neural networks), we will need to power up our toolset.

In this notebook, we introduce the PyTorch library (aka `torch`) and give a brief introduction into how it works.

In [ ]:
import numpy as np
import torch

In [ ]:
def f(x):
    return -np.exp(-(x[0]**2 + x[1]**2))

def df(x):
    return -2*np.asarray(x)*f(x)

## Pytorch Overview

- At its heart pytorch has a few core features:
    - n-dimensional array or **tensor** (similar to numpy)
    - ability to execute on specialized hardware (like GPU or TPU)
    - rich set of functions useful for computation, machine learning, and deep learning
    - support for deploying ML models to production and running on *edge* devices
    - *automatic differentiation*
- This last point is one we'll focus on...

### Basic numpy-like functionality


Many of our familiar numpy array creation functions have been replicated in pytorch

In [ ]:
torch.linspace(-2, 2, 21)

In [ ]:
torch.ones((10, 10))

In [ ]:
torch.zeros((2, 3, 2, 2))

In [ ]:
x1 = torch.eye(3)
x2 = torch.reshape(torch.arange(1, 10), (3,3))

torch.sin(x1) + torch.exp(x2) * (x1+1) / (x2**2)

We can also mix and match numpy arrays and pytorch tensors

In [ ]:
x1np = x1.numpy()
print(type(x1np))
x1np

In [ ]:
x1 + x1np

In [ ]:
np.sin(x1)

In [ ]:
# we can get a numpy array  using .numpy() method
x1.numpy()

In [ ]:
def f_torch(x):
    return -torch.exp(-(x[0]**2 + x[1]**2))

### Automatic differentiation in PyTorch

When we create a PyTorch tensor, we can specify that the gradient should (or should not) be tracked.

In [ ]:
# create our tensor with requires_grad=True
x0_torch = torch.tensor([0.2, -0.3], requires_grad=True)

# Do computation
val = f_torch(x0_torch)

In [ ]:
f(np.array([0.2, -0.3]))

In [ ]:
# call .backward() on val
val.backward()

In [ ]:
# see gradient in x0_torch.grad
x0_torch.grad

In [ ]:
df(np.array([0.2, -0.3]))

### Summary Steps

1. Define tensor with `x = torch.tensor(..., requires_grad=True)`
2. Do some computations with `x` and get a `final` tensor
3. Call `final.backward()`
4. Look up `x.grad`

> NOTE: you can only call `final.backward()` once. To get more gradients you need to compute `final` again.

## Building gradient Descent with pytorch

Let's now use pytorch + autodiff to implement gradient descent

In [ ]:
import torch

def grad_desc_torch(f, x0, epsilon=1e-3, T=200, alpha=0.1):
    trace = []
    # x is a "leaf" tensor, which is what we want
    x = torch.tensor(x0, requires_grad=True)

    for i in range(T):
        # 1. Zero the gradient from the previous step
        #    Do this BEFORE the forward pass
        if x.grad is not None:
            x.grad.zero_()

        # 2. Forward pass: compute the function value
        fx = f(x)

        # 3. Backward pass: compute the gradient
        fx.backward()
        dfdx = x.grad
        err = max(abs(dfdx))
        
        # --- Store results before updating x ---
        status = dict(
            i=i,
            err=err.item(),
            x=x.clone().detach().numpy(),
            fx=fx.item(), # fx is a scalar, so .item() is cleaner
            dfdx=dfdx.clone().detach().numpy(),
        )
        trace.append(status)

        if err < epsilon:
            return trace

        # 4. Update step: modify x without tracking it in the graph
        with torch.no_grad():
            x -= alpha * dfdx # In-place update is efficient

    raise ValueError("No convergence")

In [ ]:
trace_torch = grad_desc_torch(f_torch, [2, -0.3])
trace_torch[-1]